<a href="https://colab.research.google.com/github/muneebrabbani/computer-parts-identifier/blob/main/TriageBot_Breakout_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 TriageBot — Self-Reviewing Support Agent for NimbusPay

**Frameworks:** LangChain (LLM + Tools) · LangGraph (Control Flow + Redo Loop)

### Workflow at a glance
```
[START] → triage_node ──┬── faq_node ──────────────────────┐
                         ├── tool_node ── (over-cap?) ──────┤
                         └── escalate_node ─────────────────┤
                                                            ▼
                                                      draft_node
                                                            │
                                                      review_node
                                                       │        │
                                                    PASS      FAIL (retry ≤3)
                                                       │        │
                                                     [END]  draft_node
```

| Responsibility | Library |
|---|---|
| LLM calls, prompt templates, `@tool` definitions | **LangChain** |
| State machine, branching, redo loop | **LangGraph** |

---
**Run order:** Execute every cell top-to-bottom.  
Jump straight to **§9 Demo** or **§10 Interactive** after setup.


## §1 · Install Dependencies

In [1]:
# Install required packages
# (Colab restarts kernel automatically if needed — just re-run from top)
%pip install -q langchain langchain-openai langchain-core langgraph openai
print("✅ Packages installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 22.3 MB/s eta 0:00:00
✅ Packages installed


## §2 · API Key Setup

Your OpenAI key is loaded from **Colab Secrets** (🔑 icon in the left sidebar).  
Add a secret named `OPENAI_API_KEY` — it is never stored in the notebook file.


In [2]:
import os

# ── Preferred: Colab Secrets (key icon in left sidebar) ──────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # ── Fallback: paste key directly (remove before sharing the notebook) ──
    os.environ["OPENAI_API_KEY"] = "sk-..."          # ← replace if needed
    print("⚠️  Using hardcoded key — remove before sharing!")

assert os.environ.get("OPENAI_API_KEY", "").startswith("sk-"),     "❌ No valid API key found. Add it to Colab Secrets or paste it above."


✅ API key loaded from Colab Secrets


## §3 · Imports

In [3]:
import os, re, random
from typing import Annotated, TypedDict, Literal

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

print("✅ All imports OK")


✅ All imports OK


## §4 · Configuration

In [4]:
MODEL_NAME         = "gpt-4o-mini"   # swap to "gpt-4o" for higher quality
REFUND_AUTO_CAP    = 50.00           # refunds ≤ this are auto-eligible; > escalates
MAX_REVIEW_RETRIES = 3               # redo loop hard cap — prevents infinite spin

print(f"Model          : {MODEL_NAME}")
print(f"Refund cap     : ${REFUND_AUTO_CAP:.2f}")
print(f"Max redo loops : {MAX_REVIEW_RETRIES}")


Model          : gpt-4o-mini
Refund cap     : $50.00
Max redo loops : 3


## §5 · Synthetic Data

NimbusPay's mini FAQ and a small mock accounts table.  
No real customer data is used anywhere in this notebook.


In [5]:
# 8-entry FAQ ─────────────────────────────────────────────────────────────────
FAQ_DATA = [
    {"q": "Why was I charged a $2 fee?",
     "a": "NimbusPay charges a $2 maintenance fee each month if your balance drops below $10."},
    {"q": "How do I reset my password?",
     "a": "Go to Settings → Security → Reset Password. A link is sent to your registered email."},
    {"q": "What is NimbusPay's refund policy?",
     "a": "Refunds under $50 are processed automatically within 24 hours. "
          "Amounts over $50 require a human agent review."},
    {"q": "How long does a transfer take?",
     "a": "Domestic transfers settle within 1 business day. International transfers take 3–5 days."},
    {"q": "Is my money insured?",
     "a": "Yes. NimbusPay balances are FDIC-insured up to $250,000."},
    {"q": "What currencies are supported?",
     "a": "NimbusPay supports USD, EUR, GBP, and PKR."},
    {"q": "How do I close my account?",
     "a": "Contact support@nimbuspay.com. Account closure takes up to 5 business days."},
    {"q": "Are there withdrawal limits?",
     "a": "Daily ATM withdrawal limit is $500. Online transfers are capped at $2,000/day."},
]

# Synthetic accounts ──────────────────────────────────────────────────────────
ACCOUNTS_TABLE = {
    "4821": {"name": "Aisha Malik",  "balance": 342.17,  "last_tx": "Transfer IN  $50.00"},
    "3390": {"name": "Bilal Ahmed",  "balance":   8.45,  "last_tx": "Fee charge   -$2.00"},
    "7712": {"name": "Sara Khan",    "balance": 1205.00, "last_tx": "Transfer OUT -$200.00"},
    "0001": {"name": "Demo User",    "balance":   0.00,  "last_tx": "Account opened"},
}

print(f"✅ FAQ entries   : {len(FAQ_DATA)}")
print(f"✅ Mock accounts : {len(ACCOUNTS_TABLE)}")


✅ FAQ entries   : 8
✅ Mock accounts : 4


## §6 · LangChain Tools

Four tools wired with `@tool`.  The LLM reads the docstrings to decide when to call each one.

| Tool | Trigger |
|---|---|
| `faq_lookup` | General policy / billing questions |
| `account_lookup` | Customer asks about a specific account |
| `refund_calculator` | Customer requests a refund amount |
| `open_ticket` | Bonus: complex issues needing a ticket |


In [6]:
@tool
def faq_lookup(query: str) -> str:
    """
    Search the NimbusPay FAQ for an answer matching the customer query.
    Uses keyword overlap scoring. Returns the best match or a not-found notice.

    Args:
        query: Customer question in natural language.
    Returns:
        Matching FAQ answer string, or a not-found message.
    """
    query_tokens = set(re.sub(r"[^a-z0-9 ]", "", query.lower()).split())
    best_score, best_answer = 0, None
    for entry in FAQ_DATA:
        overlap = len(query_tokens & set(re.sub(r"[^a-z0-9 ]", "", entry["q"].lower()).split()))
        if overlap > best_score:
            best_score, best_answer = overlap, entry["a"]
    return f"[FAQ] {best_answer}" if best_answer else "[FAQ] No match found. Consider escalating."


@tool
def account_lookup(account_id: str) -> str:
    """
    Look up a NimbusPay account by ID. Returns holder name, balance, last transaction.
    Only uses synthetic demo data — no real accounts.

    Args:
        account_id: Numeric account identifier (e.g. '4821').
    Returns:
        Formatted account summary, or error string if not found.
    """
    rec = ACCOUNTS_TABLE.get(account_id.strip())
    if not rec:
        return f"[ACCOUNT] Account '{account_id}' not found."
    return (f"[ACCOUNT] Account {account_id} | Holder: {rec['name']} | "
            f"Balance: ${rec['balance']:.2f} | Last transaction: {rec['last_tx']}")


@tool
def refund_calculator(amount: float) -> str:
    """
    Evaluate whether a refund is auto-eligible or must escalate to a human.
    Business rule: ≤ $50 auto-approved; > $50 requires human review.
    This tool NEVER executes a refund — it only returns the eligibility decision.

    Args:
        amount: Requested refund amount in USD.
    Returns:
        Eligibility verdict string.
    """
    if amount <= REFUND_AUTO_CAP:
        return (f"[REFUND] ${amount:.2f} is within the auto-approval limit "
                f"(≤${REFUND_AUTO_CAP:.2f}). Refund is auto-eligible. "
                "Processing begins within 24 hours.")
    return (f"[REFUND] ${amount:.2f} exceeds the auto-approval cap "
            f"(${REFUND_AUTO_CAP:.2f}). This MUST be escalated to a human agent.")


@tool
def open_ticket(summary: str, category: str = "general") -> str:
    """
    (Bonus) Open a support ticket and return a mock ticket ID.

    Args:
        summary:  Short description of the issue.
        category: Ticket category (billing / account / refund / general).
    Returns:
        Confirmation string with mock ticket number.
    """
    ticket_id = f"NP-{random.randint(10000, 99999)}"
    return (f"[TICKET] {ticket_id} opened. Category: {category}. "
            f"Summary: '{summary}'. Agent will follow up within 2 business hours.")


# Tool registry used by nodes
ALL_TOOLS = [faq_lookup, account_lookup, refund_calculator, open_ticket]
TOOL_MAP  = {t.name: t for t in ALL_TOOLS}

print("✅ Tools registered:", [t.name for t in ALL_TOOLS])


✅ Tools registered: ['faq_lookup', 'account_lookup', 'refund_calculator', 'open_ticket']


## §7 · LLM & Prompts

In [7]:
# LLM instance (temperature=0 → deterministic routing)
llm            = ChatOpenAI(model=MODEL_NAME, temperature=0,
                            openai_api_key=os.environ["OPENAI_API_KEY"])
llm_with_tools = llm.bind_tools(ALL_TOOLS)   # tool-aware variant for tool_node

# ── System prompts ────────────────────────────────────────────────────────────

TRIAGE_SYSTEM = """You are the triage module for NimbusPay support.
Given a customer message, respond with EXACTLY one word (lowercase):
  faq       – answerable from FAQ (fees, policies, password, transfers)
  tool      – needs account lookup, refund calculation, or ticket creation
  escalate  – over-cap money action, abusive, or too complex for the bot
Reply with only the single word. No explanation."""

DRAFT_SYSTEM = """You are TriageBot, a polite NimbusPay support agent.
Using the tool results provided, write a concise helpful reply.
If a refund exceeds $50, state it must be escalated — do NOT promise to process it."""

REVIEW_SYSTEM = f"""You are the quality-review module for TriageBot.
Check the draft against these rules:
  R1. Must NOT promise auto-processing of a refund above ${REFUND_AUTO_CAP:.2f}.
  R2. Must NOT leak internal account IDs beyond what the customer provided.
  R3. Must NOT be rude or dismissive.
  R4. Must directly address the customer's question.
Reply EXACTLY with:
  PASS              – draft is acceptable
  FAIL: <reason>    – draft breaks a rule"""

print("✅ LLM ready:", MODEL_NAME)
print("✅ Prompts defined: TRIAGE_SYSTEM, DRAFT_SYSTEM, REVIEW_SYSTEM")


✅ LLM ready: gpt-4o-mini
✅ Prompts defined: TRIAGE_SYSTEM, DRAFT_SYSTEM, REVIEW_SYSTEM


## §8 · LangGraph State Machine

### State
`BotState` is the single shared dict that every node reads from and writes to.

### Nodes
| Node | Role |
|---|---|
| `triage_node` | Classifies message → faq / tool / escalate |
| `faq_node` | Runs `faq_lookup` tool |
| `tool_node` | Lets LLM pick & run the right tool(s) |
| `draft_node` | Writes customer-facing reply from tool results |
| `review_node` | Checks draft against R1–R4; triggers redo loop on fail |
| `escalate_node` | Returns "handing to human agent" without drafting |

### Edges
Conditional edges implement all branching and the redo loop.


In [8]:
# ── Graph State ───────────────────────────────────────────────────────────────

class BotState(TypedDict):
    messages        : Annotated[list, add_messages]  # full turn history
    customer_message: str          # raw input (kept for routing & prompts)
    route           : str          # 'faq' | 'tool' | 'escalate'
    draft_reply     : str          # current draft before review
    final_reply     : str          # approved reply sent to customer
    review_retries  : int          # redo-loop counter
    review_passed   : bool         # True once review approves the draft
    tool_results    : list[str]    # accumulated tool outputs for this turn


# ── Nodes ─────────────────────────────────────────────────────────────────────

def triage_node(state: BotState) -> dict:
    """Classify message → 'faq' | 'tool' | 'escalate'."""
    resp  = llm.invoke([SystemMessage(content=TRIAGE_SYSTEM),
                        HumanMessage(content=state["customer_message"])])
    route = resp.content.strip().lower().split()[0]
    if route not in ("faq", "tool", "escalate"):
        route = "escalate"   # safe fallback
    print(f"  [TRIAGE] → {route.upper()}")
    return {"route": route}


def faq_node(state: BotState) -> dict:
    """Run faq_lookup and store result for draft_node."""
    result = faq_lookup.invoke({"query": state["customer_message"]})
    print(f"  [FAQ TOOL] {result}")
    return {"tool_results": [result]}


def tool_node(state: BotState) -> dict:
    """Let LLM choose tool(s), execute them, collect results."""
    resp = llm_with_tools.invoke([
        SystemMessage(content=("You are a NimbusPay tool-use agent. "
                               "Call the appropriate tool(s). Do not reply in prose yet.")),
        HumanMessage(content=state["customer_message"]),
    ])
    results = []
    if hasattr(resp, "tool_calls") and resp.tool_calls:
        for call in resp.tool_calls:
            fn = TOOL_MAP.get(call["name"])
            if fn:
                res = fn.invoke(call["args"])
                print(f"  [TOOL: {call['name']}] {res}")
                results.append(res)
    else:
        results.append("[TOOL] No tool call made.")
    return {"tool_results": results}


def draft_node(state: BotState) -> dict:
    """Generate draft customer reply from tool results."""
    context = "\n".join(state.get("tool_results", []))
    resp    = llm.invoke([
        SystemMessage(content=DRAFT_SYSTEM),
        HumanMessage(content=(f"Customer message: {state['customer_message']}\n\n"
                               f"Tool results:\n{context}\n\nWrite your reply now.")),
    ])
    draft = resp.content.strip()
    print(f"  [DRAFT] {draft}")
    return {"draft_reply": draft}


def escalate_node(state: BotState) -> dict:
    """Return standard escalation message; skip review."""
    reply = ("I'm sorry, but this request requires a human agent review. "
             "I'm escalating your case now — a NimbusPay agent will contact you "
             "within 2 business hours. Thank you for your patience.")
    print("  [ESCALATE] Handing off to human agent.")
    return {"draft_reply": reply, "final_reply": reply, "review_passed": True}


def review_node(state: BotState) -> dict:
    """Check draft against R1–R4. Approve or feed failure reason back."""
    retries = state.get("review_retries", 0)
    resp    = llm.invoke([
        SystemMessage(content=REVIEW_SYSTEM),
        HumanMessage(content=(f"Customer message: {state['customer_message']}\n\n"
                               f"Draft reply:\n{state['draft_reply']}")),
    ])
    verdict = resp.content.strip()
    passed  = verdict.upper().startswith("PASS")
    print(f"  [REVIEW #{retries + 1}] {verdict}")

    if passed:
        return {"review_passed": True, "review_retries": retries + 1,
                "final_reply": state["draft_reply"]}
    return {"review_passed": False, "review_retries": retries + 1,
            "tool_results": state.get("tool_results", []) + [f"[REVIEW FEEDBACK] {verdict}"]}


# ── Routing functions (conditional edges) ────────────────────────────────────

def route_initial(state) -> Literal["faq_node", "tool_node", "escalate_node"]:
    return {"faq": "faq_node", "tool": "tool_node",
            "escalate": "escalate_node"}.get(state["route"], "escalate_node")

def route_after_faq(state) -> Literal["draft_node"]:
    return "draft_node"

def route_after_tool(state) -> Literal["draft_node", "escalate_node"]:
    return "escalate_node" if any("MUST be escalated" in r
                                  for r in state.get("tool_results", [])) else "draft_node"

def route_after_review(state) -> Literal["END", "draft_node"]:
    if state["review_passed"] or state.get("review_retries", 0) >= MAX_REVIEW_RETRIES:
        if not state["review_passed"]:
            print(f"  [REVIEW] Max retries reached — accepting last draft.")
        return "END"
    return "draft_node"


# ── Assemble & compile the graph ─────────────────────────────────────────────

def build_graph():
    g = StateGraph(BotState)
    g.add_node("triage_node",   triage_node)
    g.add_node("faq_node",      faq_node)
    g.add_node("tool_node",     tool_node)
    g.add_node("draft_node",    draft_node)
    g.add_node("escalate_node", escalate_node)
    g.add_node("review_node",   review_node)

    g.add_edge(START, "triage_node")
    g.add_conditional_edges("triage_node",  route_initial,
        {"faq_node": "faq_node", "tool_node": "tool_node", "escalate_node": "escalate_node"})
    g.add_conditional_edges("faq_node",     route_after_faq,    {"draft_node": "draft_node"})
    g.add_conditional_edges("tool_node",    route_after_tool,
        {"draft_node": "draft_node", "escalate_node": "escalate_node"})
    g.add_edge("draft_node", "review_node")
    g.add_conditional_edges("review_node",  route_after_review, {"END": END, "draft_node": "draft_node"})
    g.add_edge("escalate_node", END)
    return g.compile()

graph = build_graph()
print("✅ LangGraph compiled successfully")


✅ LangGraph compiled successfully


## §9 · Runner Helper

In [9]:
def run(customer_message: str) -> str:
    """
    Run TriageBot on a single customer message and return the final reply.

    Args:
        customer_message: Raw text input from the customer.
    Returns:
        Approved reply string.
    """
    print("\n" + "═" * 60)
    print(f"  CUSTOMER : {customer_message}")
    print("═" * 60)

    result = graph.invoke({
        "messages"        : [HumanMessage(content=customer_message)],
        "customer_message": customer_message,
        "route"           : "",
        "draft_reply"     : "",
        "final_reply"     : "",
        "review_retries"  : 0,
        "review_passed"   : False,
        "tool_results"    : [],
    })

    reply = result.get("final_reply") or result.get("draft_reply") or "(no reply)"
    print(f"\n  TRIAGEBOT: {reply}")
    return reply

print("✅ run() helper ready")


✅ run() helper ready


## §10 · Demo — Required Cases

Four cases covering every required proof point:

| # | Message | Expected path |
|---|---|---|
| 1 | `"Why was I charged a $2 fee?"` | FAQ → draft → review PASS → END |
| 2 | `"What's the balance on account 4821?"` | tool (account_lookup) → draft → review PASS → END |
| 3 | `"I want a $120 refund."` | tool (refund_calculator) → **escalate** → END |
| 4 | `"Please confirm my $200 refund will be auto-processed."` | tool → draft → **review FAIL** → redo → PASS → END |


In [10]:
DEMO_MESSAGES = [
    "Why was I charged a $2 fee?",                             # Case 1: FAQ branch
    "What's the balance on account 4821?",                     # Case 2: Tool → account lookup
    "I want a $120 refund.",                                   # Case 3: Over-cap → escalate
    "Please confirm my $200 refund will be auto-processed.",   # Case 4: Review loop trigger
]

print("█" * 60)
print("  TRIAGEBOT DEMO  –  NimbusPay Support Agent")
print("█" * 60)

for msg in DEMO_MESSAGES:
    run(msg)

print("\n" + "█" * 60)
print("  DEMO COMPLETE")
print("█" * 60)


████████████████████████████████████████████████████████████
  TRIAGEBOT DEMO  –  NimbusPay Support Agent
████████████████████████████████████████████████████████████

════════════════════════════════════════════════════════════
  CUSTOMER : Why was I charged a $2 fee?
════════════════════════════════════════════════════════════
  [TRIAGE] → FAQ
  [FAQ TOOL] [FAQ] NimbusPay charges a $2 maintenance fee each month if your balance drops below $10.
  [DRAFT] Hello! The $2 fee you were charged is a maintenance fee that applies when your balance drops below $10. If you have any further questions or need assistance, feel free to ask!
  [REVIEW #1] PASS

  TRIAGEBOT: Hello! The $2 fee you were charged is a maintenance fee that applies when your balance drops below $10. If you have any further questions or need assistance, feel free to ask!

════════════════════════════════════════════════════════════
  CUSTOMER : What's the balance on account 4821?
════════════════════════════════════════════

## §11 · Interactive — Try Your Own Message

Type any NimbusPay support message and run the cell.


In [15]:
# ── Change this message and run the cell ─────────────────────────────────────
YOUR_MESSAGE = "what is my account balance, my account ID is 0001 ?"

run(YOUR_MESSAGE)



════════════════════════════════════════════════════════════
  CUSTOMER : what is my account balance, my account ID is 0001 ?
════════════════════════════════════════════════════════════
  [TRIAGE] → TOOL
  [TOOL: account_lookup] [ACCOUNT] Account 0001 | Holder: Demo User | Balance: $0.00 | Last transaction: Account opened
  [DRAFT] Hello! Thank you for reaching out. Your current account balance is $0.00. If you have any further questions or need assistance, feel free to ask!
  [REVIEW #1] PASS

  TRIAGEBOT: Hello! Thank you for reaching out. Your current account balance is $0.00. If you have any further questions or need assistance, feel free to ask!


'Hello! Thank you for reaching out. Your current account balance is $0.00. If you have any further questions or need assistance, feel free to ask!'